# Однофакторный анализ: локальный рецидив рака ротоглотки

Исходная таблица — `w4/analytic_table.csv` (результат недели 4, признаки в исходных единицах), файл не изменяется.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

In [2]:
data = pd.read_csv("../w4/analytic_table.csv")
target = "Target_local_recurrence"

data.shape, data[target].sum()

((140, 33), np.int64(12))

In [3]:
# признак: группа сравнения
binary = {"Gender": "Female", "Induction_Chemotherapy": "N", "Concurrent_chemotherapy": "N"}
nominal = {"Race": "White", "Tumor_side": "L", "Tumor_subsite": "BOT", "Smoking_status_at_diagnosis": "Never"}

# признак: категории по порядку; группа сравнения — самая многочисленная категория
ordinal = {
    "T_category": [1, 2, 3, 4],
    "N_category": ["0", "1", "2a", "2b", "2c", "3"],
    "AJCC_Stage": ["I", "II", "III", "IV"],
    "Pathological_grade": ["I", "II", "II-III", "III", "IV"],
}
ordinal_reference = {"T_category": 2, "N_category": "2b", "AJCC_Stage": "IV", "Pathological_grade": "III"}

quant = [col for col in data.columns if col not in ["Patient_ID", target, *binary, *nominal, *ordinal]]

len(binary), len(nominal), len(ordinal), len(quant)

(3, 4, 4, 20)

## Меры связи для категориальных признаков

In [4]:
def ratio_measures(a, b, c, d):
    """a, b — исход 1 и исход 0 в группе; c, d — в группе сравнения."""
    rd = a / (a + b) - c / (c + d)
    rr = a / (a + b) / (c / (c + d)) if c > 0 else None
    odds_ratio = a * d / (b * c) if b * c > 0 else None
    rr_ci = or_ci = None
    if min(a, b, c, d) > 0:
        z = 1.96
        se_rr = np.sqrt(1 / a - 1 / (a + b) + 1 / c - 1 / (c + d))
        se_or = np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
        rr_ci = np.exp(np.log(rr) + np.array([-z, z]) * se_rr)
        or_ci = np.exp(np.log(odds_ratio) + np.array([-z, z]) * se_or)
    return rd, rr, rr_ci, odds_ratio, or_ci


def show(value):
    return "не вычисляется" if value is None else f"{value:.2f}"


def show_ci(ci):
    return "не вычисляется" if ci is None else f"{ci[0]:.2f}–{ci[1]:.2f}"


def note(a, b, c, total):
    notes = []
    if c == 0:
        notes.append("в группе сравнения нет исходов")
    if a == 0:
        notes.append("в группе нет исходов")
    if b == 0:
        notes.append("в группе нет пациентов без исхода")
    if total < 10:
        notes.append("меньше 10 пациентов")
    return ", ".join(notes)


def univariate(col, order, reference):
    known = data[col].notna()
    counts = pd.crosstab(data.loc[known, col], data.loc[known, target]).reindex(order, fill_value=0)

    table = pd.DataFrame({"Исход 1": counts[1], "Исход 0": counts[0]})
    table["Всего"] = table["Исход 1"] + table["Исход 0"]
    table["Риск"] = (table["Исход 1"] / table["Всего"]).round(3)

    c, d = table.loc[reference, ["Исход 1", "Исход 0"]]
    measures = {}
    for group, (a, b, total) in table[["Исход 1", "Исход 0", "Всего"]].iterrows():
        if group == reference:
            measures[group] = ["—"] * 5 + ["группа сравнения" + (", меньше 10 пациентов" if total < 10 else "")]
            continue
        rd, rr, rr_ci, odds_ratio, or_ci = ratio_measures(a, b, c, d)
        measures[group] = [f"{rd:+.3f}", show(rr), show_ci(rr_ci), show(odds_ratio), show_ci(or_ci), note(a, b, c, total)]
    columns = ["RD", "RR", "RR 95% ДИ", "OR", "OR 95% ДИ", "Примечание"]
    table = table.join(pd.DataFrame.from_dict(measures, orient="index", columns=columns))

    overall = table[["Исход 1", "Исход 0", "Всего"]].sum()
    table.loc["Всего"] = [*overall, round(overall["Исход 1"] / overall["Всего"], 3)] + [""] * len(columns)
    table.index = table.index.astype(str)
    table.index.name = col

    print(f"{col}: группа сравнения — {reference}")
    if (~known).sum():
        print(f"пропуск признака у {(~known).sum()} пациентов, они не входят в таблицу")
    display(table)

## Бинарные признаки

RD — разность рисков, RR — отношение рисков, OR — отношение шансов группы относительно группы сравнения, 95% ДИ — доверительный интервал Вальда.

In [5]:
for col, reference in binary.items():
    univariate(col, [reference] + [g for g in data[col].dropna().unique() if g != reference], reference)

Gender: группа сравнения — Female


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Gender,,,,,,,,,,
Female,1,22,23,0.043,—,—,—,—,—,группа сравнения
Male,11,106,117,0.094,+0.051,2.16,0.29–15.94,2.28,0.28–18.61,
Всего,12,128,140,0.086,,,,,,


Induction_Chemotherapy: группа сравнения — N


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Induction_Chemotherapy,,,,,,,,,,
N,8,84,92,0.087,—,—,—,—,—,группа сравнения
Y,4,44,48,0.083,-0.004,0.96,0.30–3.02,0.95,0.27–3.35,
Всего,12,128,140,0.086,,,,,,


Concurrent_chemotherapy: группа сравнения — N


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Concurrent_chemotherapy,,,,,,,,,,
N,1,38,39,0.026,—,—,—,—,—,группа сравнения
Y,11,90,101,0.109,+0.083,4.25,0.57–31.81,4.64,0.58–37.25,
Всего,12,128,140,0.086,,,,,,


## Номинальные признаки

In [6]:
for col, reference in nominal.items():
    others = data[col].value_counts().drop(reference).index.tolist()  # по убыванию числа пациентов
    univariate(col, [reference] + others, reference)

Race: группа сравнения — White


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Race,,,,,,,,,,
White,11,120,131,0.084,—,—,—,—,—,группа сравнения
Hispanic,1,4,5,0.200,+0.116,2.38,0.38–15.03,2.73,0.28–26.58,меньше 10 пациентов
Black,0,3,3,0.000,-0.084,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
Asian,0,1,1,0.000,-0.084,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
Всего,12,128,140,0.086,,,,,,


Tumor_side: группа сравнения — L


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Tumor_side,,,,,,,,,,
L,6,68,74,0.081,—,—,—,—,—,группа сравнения
R,6,60,66,0.091,+0.010,1.12,0.38–3.31,1.13,0.35–3.70,
Всего,12,128,140,0.086,,,,,,


Tumor_subsite: группа сравнения — BOT


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Tumor_subsite,,,,,,,,,,
BOT,4,63,67,0.060,—,—,—,—,—,группа сравнения
Tonsil,5,46,51,0.098,+0.038,1.64,0.46–5.81,1.71,0.44–6.73,
GPS,0,10,10,0.000,-0.060,0.00,не вычисляется,0.00,не вычисляется,в группе нет исходов
Other,1,7,8,0.125,+0.065,2.09,0.27–16.51,2.25,0.22–23.04,меньше 10 пациентов
Soft_palate,2,1,3,0.667,+0.607,11.17,3.22–38.68,31.50,2.33–426.05,меньше 10 пациентов
Pharyngeal_wall,0,1,1,0.000,-0.060,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
Всего,12,128,140,0.086,,,,,,


Smoking_status_at_diagnosis: группа сравнения — Never


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Smoking_status_at_diagnosis,,,,,,,,,,
Never,2,41,43,0.047,—,—,—,—,—,группа сравнения
Former,5,49,54,0.093,+0.046,1.99,0.41–9.76,2.09,0.39–11.35,
Current,5,38,43,0.116,+0.070,2.50,0.51–12.19,2.70,0.49–14.74,
Всего,12,128,140,0.086,,,,,,


## Порядковые признаки

In [7]:
for col, order in ordinal.items():
    univariate(col, order, ordinal_reference[col])

T_category: группа сравнения — 2


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
T_category,,,,,,,,,,
1,0,33,33,0.000,-0.105,0.00,не вычисляется,0.00,не вычисляется,в группе нет исходов
2,6,51,57,0.105,—,—,—,—,—,группа сравнения
3,3,25,28,0.107,+0.002,1.02,0.27–3.77,1.02,0.24–4.42,
4,3,19,22,0.136,+0.031,1.30,0.35–4.73,1.34,0.30–5.91,
Всего,12,128,140,0.086,,,,,,


N_category: группа сравнения — 2b


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
N_category,,,,,,,,,,
0,0,10,10,0.000,-0.072,0.00,не вычисляется,0.00,не вычисляется,в группе нет исходов
1,1,17,18,0.056,-0.017,0.77,0.10–6.16,0.75,0.08–6.88,
2a,0,10,10,0.000,-0.072,0.00,не вычисляется,0.00,не вычисляется,в группе нет исходов
2b,5,64,69,0.072,—,—,—,—,—,группа сравнения
2c,6,25,31,0.194,+0.121,2.67,0.88–8.09,3.07,0.86–10.98,
3,0,2,2,0.000,-0.072,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
Всего,12,128,140,0.086,,,,,,


AJCC_Stage: группа сравнения — IV


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
AJCC_Stage,,,,,,,,,,
I,0,2,2,0.000,-0.096,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
II,0,4,4,0.000,-0.096,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
III,1,18,19,0.053,-0.043,0.55,0.08–4.02,0.53,0.06–4.32,
IV,11,104,115,0.096,—,—,—,—,—,группа сравнения
Всего,12,128,140,0.086,,,,,,


Pathological_grade: группа сравнения — III
пропуск признака у 16 пациентов, они не входят в таблицу


,Исход 1,Исход 0,Всего,Риск,RD,RR,RR 95% ДИ,OR,OR 95% ДИ,Примечание
Pathological_grade,,,,,,,,,,
I,0,1,1,0.000,-0.065,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
II,6,47,53,0.113,+0.049,1.75,0.52–5.89,1.85,0.49–6.95,
II-III,0,6,6,0.000,-0.065,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
III,4,58,62,0.065,—,—,—,—,—,группа сравнения
IV,0,2,2,0.000,-0.065,0.00,не вычисляется,0.00,не вычисляется,"в группе нет исходов, меньше 10 пациентов"
Всего,10,114,124,0.081,,,,,,


## Коэффициент V Крамера

In [8]:
def cramers_v(col):
    counts = pd.crosstab(data[col], data[target]).to_numpy()  # пропуски признака не учитываются
    n = counts.sum()
    expected = counts.sum(axis=1, keepdims=True) * counts.sum(axis=0, keepdims=True) / n
    chi2 = ((counts - expected) ** 2 / expected).sum()
    return n, len(counts), np.sqrt(chi2 / (n * (min(counts.shape) - 1))), (expected < 5).mean()


kinds = {**dict.fromkeys(binary, "бинарный"), **dict.fromkeys(nominal, "номинальный"),
         **dict.fromkeys(ordinal, "порядковый")}
cramer = pd.DataFrame([cramers_v(col) for col in kinds], index=list(kinds),
                      columns=["пациентов", "групп", "V Крамера", "доля ячеек с ожидаемой частотой < 5"])
cramer.insert(0, "тип", pd.Series(kinds))
cramer.round(3).sort_values("V Крамера", ascending=False)

,тип,пациентов,групп,V Крамера,доля ячеек с ожидаемой частотой < 5
Tumor_subsite,номинальный,140,6,0.325,0.583
N_category,порядковый,140,6,0.224,0.500
T_category,порядковый,140,4,0.174,0.500
Concurrent_chemotherapy,бинарный,140,2,0.133,0.250
Pathological_grade,порядковый,124,5,0.119,0.600
Smoking_status_at_diagnosis,номинальный,140,3,0.100,0.500
Race,номинальный,140,4,0.093,0.750
AJCC_Stage,порядковый,140,4,0.083,0.625
Gender,бинарный,140,2,0.067,0.250
Tumor_side,номинальный,140,2,0.018,0.000


## Количественные признаки

OR при увеличении признака на межквартильный размах (МКР = Q3 − Q1) — из однофакторной логистической регрессии по признаку в исходных единицах.

In [9]:
def logistic(x, y):
    """Логистическая регрессия y ~ 1 + x методом Ньютона, возвращает наклон и его стандартную ошибку."""
    X = np.column_stack([np.ones_like(x), x])
    beta = np.zeros(2)
    for _ in range(100):
        p = 1 / (1 + np.exp(-X @ beta))
        hessian = X.T @ (X * (p * (1 - p))[:, None])
        step = np.linalg.solve(hessian, X.T @ (y - p))
        beta += step
        if np.abs(step).max() < 1e-10:
            break
    p = 1 / (1 + np.exp(-X @ beta))
    covariance = np.linalg.inv(X.T @ (X * (p * (1 - p))[:, None]))
    return beta[1], np.sqrt(covariance[1, 1])

In [10]:
units = {
    "Age_at_diagnosis": "лет",
    "Smoking_Pack-Years": "пачко-лет",
    "Total_prescribed_Radiation_treatment_dose": "Гр",
    "Dose_per_fraction": "Гр за фракцию",
    "GTVp_shape_MeshVolume": "мм³",
    "GTVp_shape_Sphericity": "безразмерный",
    "GTVp_shape_Elongation": "безразмерный",
    "GTVp_shape_Flatness": "безразмерный",
    "GTVp_firstorder_10Percentile": "HU",
    "GTVp_firstorder_90Percentile": "HU",
    "GTVp_firstorder_Entropy": "бит",
    "GTVp_firstorder_Kurtosis": "безразмерный",
    "GTVp_firstorder_Maximum": "HU",
    "GTVp_firstorder_Minimum": "HU",
    "GTVp_glcm_Correlation": "безразмерный",
    "GTVp_glcm_InverseVariance": "безразмерный",
    "GTVp_gldm_DependenceNonUniformityNormalized": "безразмерный",
    "GTVp_glszm_SizeZoneNonUniformityNormalized": "безразмерный",
    "GTVp_ngtdm_Busyness": "усл. ед.",
    "GTVp_ngtdm_Contrast": "усл. ед.",
}

def show_number(value):
    return np.format_float_positional(value, precision=4, unique=False, fractional=False, trim="-")


rows = {}
for col in quant:
    known = data[col].notna()
    x, y = data.loc[known, col].to_numpy(float), data.loc[known, target].to_numpy(float)
    q1, q3 = np.quantile(x, [0.25, 0.75])
    slope, se = logistic((x - np.median(x)) / (q3 - q1), y)  # наклон на один МКР
    ci = np.exp(slope + np.array([-1.96, 1.96]) * se)
    rows[col] = [known.sum(), int(y.sum()), units[col], show_number(q1), show_number(q3), show_number(q3 - q1),
                 round(np.exp(slope), 2), f"{ci[0]:.2f}–{ci[1]:.2f}"]

pd.DataFrame.from_dict(rows, orient="index",
                       columns=["пациентов", "исход 1", "единицы", "Q1", "Q3", "МКР", "OR на МКР", "OR 95% ДИ"])

,пациентов,исход 1,единицы,Q1,Q3,МКР,OR на МКР,OR 95% ДИ
Age_at_diagnosis,140,12,лет,52,65,13,1.55,0.67–3.58
Smoking_Pack-Years,130,11,пачко-лет,0,30.75,30.75,2.38,1.17–4.85
Total_prescribed_Radiation_treatment_dose,140,12,Гр,66,70,4,3.04,0.80–11.62
Dose_per_fraction,140,12,Гр за фракцию,2.121,2.2,0.07879,1.13,0.69–1.87
GTVp_shape_MeshVolume,140,12,мм³,4263,17750,13490,1.26,0.91–1.75
GTVp_shape_Sphericity,140,12,безразмерный,0.5459,0.6963,0.1504,0.54,0.23–1.27
GTVp_shape_Elongation,140,12,безразмерный,0.6346,0.8174,0.1828,1.86,0.76–4.54
GTVp_shape_Flatness,140,12,безразмерный,0.4473,0.6325,0.1852,1.36,0.57–3.28
GTVp_firstorder_10Percentile,139,12,HU,37,65,28,1.01,0.65–1.56
GTVp_firstorder_90Percentile,139,12,HU,101,124,23,1.06,0.79–1.41


## Выводы

Рецидив был у 12 из 140 пациентов (риск 0.086), меры неточны.

**Сильнее всего связаны с исходом** подлокализация (V Крамера 0.325) и категория N (V 0.224): риск 0.667 при опухоли мягкого нёба против 0.060 в корне языка (RR 11.17, OR 31.50), 0.194 при N2c против 0.072 при N2b (RR 2.67, ДИ 0.88–8.09). При T1 рецидивов нет (0 из 33). Среди количественных признаков 95% ДИ не включает 1 у пачко-лет (OR на МКР 2.38, 1.17–4.85) и максимальной плотности опухоли (OR 1.47, 1.03–2.09).

**Согласие с известным о болезни.** Курение, распространённость по T и N и локализация вне миндалины и корня языка, где опухоль реже ВПЧ-положительная, ухудшают локальный контроль, связи с этим согласуются. Для максимальной плотности ясного объяснения нет.

**Связи, которые может объяснять другой признак.** Одновременная химиотерапия (RR 4.25, 0.57–31.81) и полная доза (OR 3.04 на 4 Гр) назначаются при распространённой опухоли, их связь может объясняться стадией. Пачко-годы и статус курения отражают одно воздействие.

**Малое число наблюдений.** Мера для мягкого нёба получена по 3 пациентам (ДИ OR 2.33–426.05), ещё 11 групп меньше 10 пациентов отмечены в таблицах, в группах без исходов RR и OR равны 0. У всех категориальных признаков, кроме стороны опухоли, ожидаемая частота меньше 5 в 25–75% ячеек.